# Row Echelon Form of a Matrix

*Course notes for **Math for Machine Learning**, C1 · W2 · L2 — "Row Echelon Form of a Matrix" (DeepLearning.AI).*

The previous lecture said: *rank = number of non-zero rows in row echelon form*. This one shows the **step-by-step procedure** to reach REF, and how to read the rank straight off the result — by counting the **1s on the diagonal** (the pivots).

**The recipe (forward elimination):**
1. Divide each row by its **leftmost (pivot) coefficient** so it starts with a 1.
2. Subtract to **clear the entries below** that pivot.
3. Move to the next row and repeat on the **leftmost non-zero coefficient**.

In [1]:
import numpy as np
np.set_printoptions(precision=2, suppress=True)

## 1. Non-singular example

$$ \begin{bmatrix} 5 & 1 \\ 4 & -3 \end{bmatrix} $$

**Step 1 — divide each row by its leftmost coefficient** (5 and 4):

$$ \begin{bmatrix} 1 & 0.2 \\ 1 & -0.75 \end{bmatrix} $$

**Step 2 — subtract row 1 from row 2** to clear below the first pivot:

$$ \begin{bmatrix} 1 & 0.2 \\ 0 & -0.95 \end{bmatrix} $$

**Step 3 — divide row 2 by its leftmost non-zero coefficient** ($-0.95$):

$$ \begin{bmatrix} 1 & 0.2 \\ 0 & 1 \end{bmatrix} \quad\text{(row echelon form).}$$

Two **1s on the diagonal** → **rank 2** → non-singular.

In [2]:
A = np.array([[5.0, 1.0], [4.0, -3.0]])

M = A.copy()
M[0] = M[0] / M[0][0]          # row 1 starts with 1
M[1] = M[1] / M[1][0]          # row 2 starts with 1
print("after step 1:\n", M)

M[1] = M[1] - M[0]             # clear below first pivot
print("\nafter step 2:\n", M)

M[1] = M[1] / M[1][1]          # scale second pivot to 1
print("\nrow echelon form:\n", M)
print("\n1s on diagonal -> rank =", int(round(np.trace(M))))

after step 1:
 [[ 1.    0.2 ]
 [ 1.   -0.75]]

after step 2:
 [[ 1.    0.2 ]
 [ 0.   -0.95]]

row echelon form:
 [[ 1.   0.2]
 [-0.   1. ]]

1s on diagonal -> rank = 2


## 2. Singular examples

**Redundant** $\begin{bmatrix}5&1\\10&2\end{bmatrix}$. Dividing each row by its leftmost coefficient gives two identical rows $\begin{bmatrix}1&0.2\\1&0.2\end{bmatrix}$; subtracting leaves a **zero row**:

$$ \begin{bmatrix} 1 & 0.2 \\ 0 & 0 \end{bmatrix} \quad\Rightarrow\quad \text{one 1 on the diagonal} \Rightarrow \textbf{rank 1}. $$

The **zero matrix** $\begin{bmatrix}0&0\\0&0\end{bmatrix}$ has no pivots at all → **rank 0**.

In [3]:
def to_ref(A):
    """Forward elimination to REF with leading 1s; returns the REF matrix."""
    M = np.array(A, dtype=float)
    rows, cols = M.shape
    pivot = 0
    for col in range(cols):
        nz = [r for r in range(pivot, rows) if abs(M[r][col]) > 1e-12]
        if not nz:
            continue
        M[[pivot, nz[0]]] = M[[nz[0], pivot]]      # swap a non-zero pivot up
        M[pivot] = M[pivot] / M[pivot][col]        # scale pivot to 1
        for r in range(pivot + 1, rows):           # clear below
            M[r] = M[r] - M[r][col] * M[pivot]
        pivot += 1
    return M

def rank_from_diagonal(A):
    ref = to_ref(A)
    return int(sum(abs(ref[i][i]) > 1e-12 for i in range(min(ref.shape))))

for name, M in {"redundant [[5,1],[10,2]]": [[5, 1], [10, 2]],
                "zero      [[0,0],[0,0]]": [[0, 0], [0, 0]]}.items():
    print(name, "-> REF:")
    print(to_ref(M))
    print("   rank =", rank_from_diagonal(M), "\n")

redundant [[5,1],[10,2]] -> REF:
[[1.  0.2]
 [0.  0. ]]
   rank = 1 

zero      [[0,0],[0,0]] -> REF:
[[0. 0.]
 [0. 0.]]
   rank = 0 



## 3. Row echelon form, singularity, and rank

Reading the REF diagonal ties everything together:

| Matrix | REF | 1s on diagonal | Rank | |
|--------|-----|----------------|------|--|
| $\begin{bmatrix}5&1\\4&-3\end{bmatrix}$ | $\begin{bmatrix}1&0.2\\0&1\end{bmatrix}$ | 2 | 2 | non-singular |
| $\begin{bmatrix}5&1\\10&2\end{bmatrix}$ | $\begin{bmatrix}1&0.2\\0&0\end{bmatrix}$ | 1 | 1 | singular |
| $\begin{bmatrix}0&0\\0&0\end{bmatrix}$ | $\begin{bmatrix}0&0\\0&0\end{bmatrix}$ | 0 | 0 | singular |

$$ \text{rank} = \#\{\text{1s on the REF diagonal}\}, \qquad \text{non-singular} \Leftrightarrow \text{every diagonal entry is 1}. $$

In [4]:
examples = {
    "[[5,1],[4,-3]]": [[5, 1], [4, -3]],
    "[[5,1],[10,2]]": [[5, 1], [10, 2]],
    "[[0,0],[0,0]]":  [[0, 0], [0, 0]],
}
for name, A in examples.items():
    rank = rank_from_diagonal(A)
    n = len(A)
    print(f"{name:16s} rank={rank}  ->  {'non-singular' if rank == n else 'singular'}"
          f"   (numpy rank {np.linalg.matrix_rank(A)})")

[[5,1],[4,-3]]   rank=2  ->  non-singular   (numpy rank 2)
[[5,1],[10,2]]   rank=1  ->  singular   (numpy rank 1)
[[0,0],[0,0]]    rank=0  ->  singular   (numpy rank 0)


## 4. Takeaways

- **Row echelon form** is reached by forward elimination: normalise each pivot to **1**, clear everything **below** it, move on.
- The **rank** is the number of **1s on the diagonal** (pivots) of the REF.
- **Non-singular ⇔ a 1 in every diagonal position** (full rank); a **zero row** (missing pivot) ⇒ singular.
- This is the practical recipe `numpy.linalg.matrix_rank` automates.

*Companion:* [the rank of a matrix in general](./C1_W2_L2_rank_in_general.ipynb) · [systems to matrices](./C1_W2_L1_systems_to_matrices.ipynb) (REF vs. RREF).